# Google Play Store Review Scraper

This notebook scrapes review text directly from the Google Play Store using the `google-play-scraper` package.

It is set up for:
- one or many app IDs
- review pagination
- de-duplication by `review_id`
- CSV export
- metadata export
- a quick validation summary after scraping

## 1) Install dependencies

Run this cell once if the environment does not already have the required packages.

In [3]:
# Uncomment if needed
!pip install -q google-play-scraper

## 2) Imports

In [6]:
import json
import time
from pathlib import Path

import pandas as pd
from google_play_scraper import Sort, reviews

## 3) Scrape settings

Update the app IDs and collection settings here.

Notes:
- `lang='en'` and `country='us'` keeps the collection focused and easier to compare.
- `sort='NEWEST'` is usually the best first pass for recent product feedback.
- Google Play paging is typically capped at 200 reviews per request.

In [14]:
APP_IDS = [
    "com.spotify.music",
    "com.instagram.android",
    "com.zhiliaoapp.musically",
    "com.snapchat.android",
    "com.whatsapp",
]

LANG = "en"
COUNTRY = "us"
SORT_NAME = "NEWEST"   # options: NEWEST, HELPFULNESS, RATING
COUNT_PER_APP = 6000      # target reviews to request per app
MAX_REVIEWS = 30000      # stop once this many unique reviews are collected total
BATCH_SIZE = 200         # per-request page size
SLEEP_SECONDS = 0.5      # small pause to reduce throttling risk

OUT_CSV = "google_play_reviews.csv"
OUT_METADATA = "google_play_reviews_run_metadata.json"

## 4) Helpers

In [15]:
SORT_MAP = {
    "NEWEST": Sort.NEWEST,
    "RATING": Sort.RATING,
    "HELPFULNESS": getattr(Sort, "MOST_RELEVANT", getattr(Sort, "HELPFULNESS", Sort.NEWEST)),
}


def normalize_review(raw, app_id, lang, country):
    content = raw.get("content") or raw.get("text") or ""
    at = raw.get("at") or raw.get("date")
    reply_content = raw.get("replyContent") or raw.get("replyText")
    reply_at = raw.get("repliedAt") or raw.get("replyDate")

    return {
        "app_id": app_id,
        "review_id": raw.get("reviewId") or raw.get("id"),
        "user_name": raw.get("userName"),
        "score": raw.get("score"),
        "review_text": content,
        "thumbs_up": raw.get("thumbsUpCount", raw.get("thumbsUp")),
        "review_created_at": at.isoformat() if hasattr(at, "isoformat") else at,
        "app_version": raw.get("reviewCreatedVersion", raw.get("version")),
        "reply_text": reply_content,
        "reply_created_at": reply_at.isoformat() if hasattr(reply_at, "isoformat") else reply_at,
        "lang": lang,
        "country": country,
    }


def scrape_one_app(
    app_id,
    lang,
    country,
    sort_name,
    count_per_app,
    batch_size,
    sleep_seconds,
    seen_review_ids,
    max_reviews_remaining,
):
    collected = []
    continuation_token = None
    requested = 0
    sort_enum = SORT_MAP[sort_name]

    while requested < count_per_app and len(collected) < max_reviews_remaining:
        this_batch = min(batch_size, count_per_app - requested, max_reviews_remaining - len(collected))

        result, continuation_token = reviews(
            app_id,
            lang=lang,
            country=country,
            sort=sort_enum,
            count=this_batch,
            continuation_token=continuation_token,
        )

        if not result:
            break

        batch_new = 0
        for raw in result:
            review_id = raw.get("reviewId") or raw.get("id")
            if review_id and review_id in seen_review_ids:
                continue
            if review_id:
                seen_review_ids.add(review_id)

            collected.append(normalize_review(raw, app_id, lang, country))
            batch_new += 1

            if len(collected) >= max_reviews_remaining:
                break

        requested += this_batch

        if continuation_token is None or batch_new == 0:
            break

        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

    return collected

## 5) Run the scrape

This cell iterates through the app basket, collects reviews, saves a CSV, and writes run metadata.

In [16]:
all_rows = []
seen_review_ids = set()
per_app_counts = {}
errors = {}

for app_id in APP_IDS:
    remaining = MAX_REVIEWS - len(all_rows)
    if remaining <= 0:
        break

    try:
        rows = scrape_one_app(
            app_id=app_id,
            lang=LANG,
            country=COUNTRY,
            sort_name=SORT_NAME,
            count_per_app=COUNT_PER_APP,
            batch_size=BATCH_SIZE,
            sleep_seconds=SLEEP_SECONDS,
            seen_review_ids=seen_review_ids,
            max_reviews_remaining=remaining,
        )
        all_rows.extend(rows)
        per_app_counts[app_id] = len(rows)
        print(f"{app_id}: collected {len(rows)} unique reviews")
    except Exception as exc:
        errors[app_id] = str(exc)
        print(f"{app_id}: error: {exc}")

df = pd.DataFrame(all_rows)
df.to_csv(OUT_CSV, index=False)

metadata = {
    "total_unique_reviews": len(df),
    "app_count_attempted": len(APP_IDS),
    "app_count_succeeded": len(per_app_counts),
    "per_app_counts": per_app_counts,
    "errors": errors,
    "lang": LANG,
    "country": COUNTRY,
    "sort": SORT_NAME,
    "count_per_app": COUNT_PER_APP,
    "max_reviews": MAX_REVIEWS,
    "output_csv": str(Path(OUT_CSV).resolve()),
}

Path(OUT_METADATA).write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(json.dumps(metadata, indent=2))

com.spotify.music: collected 6000 unique reviews
com.instagram.android: collected 6000 unique reviews
com.zhiliaoapp.musically: collected 6000 unique reviews
com.snapchat.android: collected 6000 unique reviews
com.whatsapp: collected 6000 unique reviews
{
  "total_unique_reviews": 30000,
  "app_count_attempted": 5,
  "app_count_succeeded": 5,
  "per_app_counts": {
    "com.spotify.music": 6000,
    "com.instagram.android": 6000,
    "com.zhiliaoapp.musically": 6000,
    "com.snapchat.android": 6000,
    "com.whatsapp": 6000
  },
  "errors": {},
  "lang": "en",
  "country": "us",
  "sort": "NEWEST",
  "count_per_app": 6000,
  "max_reviews": 30000,
  "output_csv": "/content/google_play_reviews.csv"
}


## 6) Quick validation

Use this to verify that the scrape worked and inspect the shape of the output.

In [17]:
df.head()

,app_id,review_id,user_name,score,review_text,thumbs_up,review_created_at,app_version,reply_text,reply_created_at,lang,country
0,com.spotify.music,89de08ec-a11a-4ea9-80d1-c1c4d2d4fc82,Eliza,5,no ads and keep it that way,0,2026-04-06T01:43:24,9.1.34.2060,None,None,en,us
1,com.spotify.music,8d3a368e-7955-4534-84ab-796d71dad11d,Kayla Sprowls,5,the best for downloading and listening to music,0,2026-04-06T01:42:43,9.1.34.2060,None,None,en,us
2,com.spotify.music,c7c4b188-5468-4046-8e0e-eb8820fb5b2f,Jenny “JMac” Sedberry,5,Spotify is a great app for listening to music ...,0,2026-04-06T01:40:29,9.1.34.2060,None,None,en,us
3,com.spotify.music,5757d546-d991-41f7-8718-9194c305b622,R “Roger That” Smith,4,feelin satisfied..!!,0,2026-04-06T01:37:45,9.1.34.2060,None,None,en,us
4,com.spotify.music,63b899be-8a0f-42cc-ac96-d89e7e80eac5,Sappala Anil Kumar,5,best 👌,0,2026-04-06T01:22:19,9.1.28.2252,None,None,en,us


In [18]:
print("Rows:", len(df))
print("Apps:", df['app_id'].nunique() if not df.empty else 0)
print("Missing review text:", int(df['review_text'].isna().sum()) if 'review_text' in df.columns else 0)
print("Duplicate review_id:", int(df['review_id'].duplicated().sum()) if 'review_id' in df.columns else 0)

if not df.empty and 'score' in df.columns:
    print("Rating distribution:")
    print(df['score'].value_counts(dropna=False).sort_index())

Rows: 30000
Apps: 5
Missing review text: 0
Duplicate review_id: 0
Rating distribution:
score
1     5920
2     1256
3     1470
4     2362
5    18992
Name: count, dtype: int64


## 7) Optional: save a cleaned version

This is just a light first-pass cleaning example.

In [19]:
if not df.empty:
    clean_df = df.copy()
    clean_df = clean_df.drop_duplicates(subset=['review_id'])
    clean_df['review_text'] = clean_df['review_text'].fillna('').astype(str).str.strip()
    clean_df = clean_df[clean_df['review_text'] != '']
    clean_df = clean_df[clean_df['score'].isin([1, 2, 3, 4, 5])]

    clean_out = "google_play_reviews_clean.csv"
    clean_df.to_csv(clean_out, index=False)
    print(f"Saved cleaned file to {Path(clean_out).resolve()}")
    print("Clean rows:", len(clean_df))
else:
    print("No rows collected yet.")

Saved cleaned file to /content/google_play_reviews_clean.csv
Clean rows: 29999


## 8) Suggested next step

Once the first scrape works, expand the app basket to 15 to 25 apps and collect 10k+ reviews total so you can run EDA on:
- rating distribution
n- review length distribution
- duplicates / repeated boilerplate
- language consistency
- missing metadata fields
- app-level balance